# Chapter 3: Assigning Roles (Role Prompting)

- [Lesson](#lesson)
- [Exercises](#exercises)
- [Example Playground](#example-playground)

## Setup

Run the cell below to load `MODEL_NAME` and define the `get_completion` helper.

In [ ]:
# Python's built-in regular expression library (used by the graders)
import re
import ollama

# Retrieve MODEL_NAME from the IPython store (set in Chapter 0).
%store -r MODEL_NAME

def get_completion(user_prompt: str, system_prompt=""):
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_prompt})
    response = ollama.chat(
        model=MODEL_NAME,
        messages=messages,
        options={"temperature": 0.0, "num_predict": 2000},
    )
    return response["message"]["content"]

MODEL_NAME

---

## Lesson

Sticking with the theme that the model only knows what you tell it, it often
helps to **ask the model to take on a specific role** — a persona, a profession,
a frame of mind — before giving it the task. This is called **role prompting**,
and the more concrete the role, the stronger the effect.

Priming a role does two useful things:

1. **It shifts style, tone, and voice.** "You are a stand-up comedian" and "You
   are a contract lawyer" will answer the same question very differently.
2. **It can improve task performance.** Telling the model to think like an
   expert in the relevant field — a mathematician, an editor, a logician —
   nudges it toward the kind of careful reasoning that role implies. It's the
   same trick that helps people: "approach this like a ______."

**Where does the role go?** Either in the `system` prompt or right in the user
turn — both work. A bonus lever is to **also tell the model who its audience
is**: "You are a physicist" reads very differently from "You are a physicist
explaining to a curious seven-year-old."


### Examples

#### Role changes the voice

Without a role, the model gives a flat, neutral answer.

In [ ]:
USER_PROMPT = "In one sentence, what do you think about squirrels?"
print(get_completion(USER_PROMPT))

Now we prime it with a role in the `SYSTEM_PROMPT`. Same question, completely different **tone, word
choice, and attitude.**

In [ ]:
# System prompt
SYSTEM_PROMPT = "You are a Shakespearean actor who answers everything in grandiose, theatrical Early Modern English."

# User prompt
USER_PROMPT = "In one sentence, what do you think about squirrels?"

print(get_completion(USER_PROMPT, SYSTEM_PROMPT))

Try the audience-context bonus too: change the system prompt to
`"You are a Shakespearean actor who answers everything in grandiose, theatrical Early Modern English. You are performing for a modern audience that struggles with archaic English, so gloss any old-fashioned phrases in plain modern terms."`
and watch the answer shift again.

#### Role can sharpen reasoning

Small models are famously bad at counting letters inside a word — ask plainly
and you'll often get the wrong number.

In [ ]:
"strawberry".count("r")

In [ ]:
# User prompt
USER_PROMPT = "How many times does the letter 'r' appear in the word 'strawberry'?"
print(get_completion(USER_PROMPT))

Now give it a role that *forces* a careful, step-by-step method. The role
isn't magic — what it really does is push the model toward spelling the word
out before answering, which is what fixes it.

**Note:** results here vary by model. A stronger model may already get it right
without the role; a weaker one may still slip even with it. The point is the
*direction* the role nudges the behaviour.

In [ ]:
# System prompt
SYSTEM_PROMPT = "You are a mathematical genius, who always shows their working"

# User prompt
USER_PROMPT = "How many times does the letter 'r' appear in the word 'strawberry'?"

print(get_completion(USER_PROMPT, SYSTEM_PROMPT))

As you'll keep seeing in this course, **many techniques can get you to the
same place** — role prompting is one tool among several. Experiment and find the
style that works for you.

Want to tinker freely? Visit the [**Example Playground**](#example-playground)
at the bottom.

#### What actually caused the fix? Let's isolate it.

The system prompt that worked did *two* things at once: it assigned a **role**
("a mathematical genius") *and* gave an **instruction** ("who always shows
their working"). When two changes ride along together and the output improves,
you can't tell which one was responsible — they're *confounded*.

To find the active ingredient, change one variable at a time and hold the rest
fixed. This is an [**ablation**](https://en.wikipedia.org/wiki/Ablation_(artificial_intelligence)). First: role, instruction removed. Then:
instruction, role removed.

In [ ]:
# System prompt
SYSTEM_PROMPT = "You are a mathematical genius"

# User prompt
USER_PROMPT = "How many times does the letter 'r' appear in the word 'strawberry'?"

print(get_completion(USER_PROMPT, SYSTEM_PROMPT))

**Role only, instruction removed → back to "twice."** The persona label on its
own did nothing for accuracy. Now do the opposite: drop the role, keep only the
instruction.

In [ ]:
# System prompt
SYSTEM_PROMPT = "You always shows your working"

# Prompt
USER_PROMPT = "How many times does the letter 'r' appear in the word 'strawberry'?"

print(get_completion(USER_PROMPT, SYSTEM_PROMPT))

#### It was the instruction, not the role.

| System prompt | Shows working? | Answer |
|---|---|---|
| *(none)* | no | 2 ✗ |
| "mathematical genius" + "shows their working" | yes | 3 ✓ |
| "mathematical genius" | no | 2 ✗ |
| "always shows your working" | yes | 3 ✓ |

The answer is right in exactly the rows where we ask the model to **show its
working** — with or without the "genius" role. The role did nothing; the
instruction did all the work.

Asking a model to work through something step by step, instead of answering
right away, is a core technique called **chain-of-thought** prompting. It often
helps on tricky questions.

One honest note: if you read the model's steps above, they're actually a bit
sloppy — it doesn't spell the word out neatly and still gets the right answer.
So don't assume the steps show exactly how it's thinking. What we can say for
sure is the practical part: **asking for the steps got us the right answer.**

**The lesson:** when a change to your prompt works, figure out *which part* of
the change was responsible by testing one thing at a time. Here it was the
instruction, not the role — easy to mix up if you'd changed both at once and
never checked.

### Why is spelling hard for language models?

Ever wondered why a model can write essays but trips over counting letters? It's
because a model doesn't read text letter by letter the way we do. It first
breaks text into **tokens** — pieces that are often several characters long.

Try it yourself: open the [Tiktokenizer app](https://tiktokenizer.vercel.app/?model=cl100k_base)
and type in `strawberry`.

You'll see the word isn't a single piece — it's split into 3 chunks `str` `aw` and `berry`, none of them is a single letter. Because the model works with these chunks
rather than individual letters, it has a hard time "seeing" the letters well
enough to count them.

That's the short version of why letter-level tasks — counting letters, reversing
a word, spotting rhymes — are a classic weak spot for language models.

---

## Exercises

### Exercise 3.1 - Changing the Voice

We just saw that a role *didn't* fix correctness — the instruction did. So what
is role prompting genuinely good for? **Controlling the style, tone, and format
of the output.** That's where roles shine.

Here the math is trivial and the model will get it right no matter what. Your
job is to change *how* it answers. Modify the `SYSTEM_PROMPT` so the model
responds **like a pirate**.

In [ ]:
# System prompt - this is the only field you should change
SYSTEM_PROMPT = "[REPLACE THIS TEXT]"

# User prompt
USER_PROMPT = "What is 2 + 2?"

response = get_completion(USER_PROMPT, SYSTEM_PROMPT)

# Function to grade exercise correctness
def grade_exercise(text):
    text = text.lower()
    pirate_words = ["arr", "ahoy", "matey", "avast", "ye", "aye", "yarr", "scurvy"]
    return ("four" in text or "4" in text) and any(word in text for word in pirate_words)

print(response)
print("\n--------------------------- GRADING ---------------------------")
print("This exercise has been solved correctly:", "✅" if grade_exercise(response) else "❌")

❓ If you want a hint, run the cell below!

In [ ]:
from hints import exercise_3_1_hint; print(exercise_3_1_hint)

### Congrats!

Once the exercise grades as solved, you've got role prompting down. On to the
next chapter!

---

## Example Playground

Experiment freely with this chapter's prompts here.

In [ ]:
USER_PROMPT = "In one sentence, what do you think about squirrels?"
print(get_completion(USER_PROMPT))

In [ ]:
SYSTEM_PROMPT = "You are a Shakespearean actor who answers everything in grandiose, theatrical Early Modern English."
USER_PROMPT = "In one sentence, what do you think about squirrels?"
print(get_completion(USER_PROMPT, SYSTEM_PROMPT))

In [ ]:
USER_PROMPT = "How many times does the letter 'r' appear in the word 'strawberry'?"
print(get_completion(USER_PROMPT))

In [ ]:
SYSTEM_PROMPT = "You are Buzz Lightyear from Toy Story"
USER_PROMPT = "What is 2 + 2?"
print(get_completion(USER_PROMPT, SYSTEM_PROMPT))